<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Selecting a Suitable Model

Model Benchmarking

Hyperparameter Tuning

Download/Export the Model

# Selecting a Suitable Model

#CRIME HOTSPOT PREDICTION - model predicts a single "Total Risk" score

## How the "One Instance" Workflow WorksWhen a user selects a crime type (e.g., "Theft") on your dashboard, the system follows this logic:Target Selection: The model focuses only on "Theft" records in your historical data.Probability Calculation: It calculates the "intensity" (0% to 100%) for every GN Division specifically for that crime.Color Mapping: The GN divisions are then colored on your heatmap:Dark Red: High Intensity ($>80\%$ probability of Theft).Yellow/Orange: Moderate Intensity ($40-60\%$).Green: Low Intensity ($<10\%$).

#Most reliable metrices
- precision
- Recall
- F1-Score
- AUC-ROC

Load the training data

In [7]:
import pandas as pd
import joblib
import json
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/DSGP/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from sklearn.metrics import (classification_report, roc_auc_score, precision_score,
                            recall_score, f1_score, confusion_matrix)

# ============================================================================
# 1. LOAD PREPROCESSED DATA
# ============================================================================
print("="*70)
print("LOADING PREPROCESSED DATA")
print("="*70)

# Load the full processed sets (Parquet preserves data types)
train_df = pd.read_parquet(os.path.join(save_path, 'train_set_processed.parquet'))
test_df = pd.read_parquet(os.path.join(save_path, 'test_set_processed.parquet'))
inference_full = pd.read_parquet(os.path.join(save_path, 'inference_data_latest.parquet'))

# Load the Scaler
scaler = joblib.load(os.path.join(save_path, 'feature_scaler.joblib'))

# Load the Feature List
with open(os.path.join(save_path, 'feature_list.json'), 'r') as f:
    features = json.load(f)

print(f"✅ Loaded {len(features)} features")
print(f"✅ Training data: {len(train_df)} rows")
print(f"✅ Test data: {len(test_df)} rows")
print(f"✅ Inference data: {len(inference_full)} rows")


LOADING PREPROCESSED DATA
✅ Loaded 44 features
✅ Training data: 2764 rows
✅ Test data: 4373 rows
✅ Inference data: 105 rows


In [9]:
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

##2. LightGBM (Gradient Boosting Machine)

In [10]:
!pip install lightgbm
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier

In [11]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import classification_report, roc_auc_score


print("\n" + "="*70)
print("PREPARING DATA FOR MODEL TRAINING")
print("="*70)

# SELECT THE SPECIFIC CRIME (Dashboard dropdown simulation)
selected_crime = 'theft'  # Change this to train different crime models
target_col = f'target_{selected_crime}_next_week'

print(f"\n🎯 Selected crime type: {selected_crime.upper()}")
print(f"🎯 Target column: {target_col}")

# Verify target column exists
if target_col not in train_df.columns:
    raise ValueError(
        f"❌ Target column '{target_col}' not found!\n"
        f"Available target columns: {[col for col in train_df.columns if 'target_' in col]}"
    )

# Extract X (features) and y (target)
X_train = train_df[features]
y_train = train_df[target_col]

X_test = test_df[features]
y_test = test_df[target_col]

X_inference = inference_full[features]

# ============================================================================
# 3. DATA VALIDATION BEFORE TRAINING
# ============================================================================
print("\n" + "="*70)
print("DATA VALIDATION BEFORE TRAINING")
print("="*70)

print(f"\n📊 Training set: {len(X_train)} rows")
print(f"📊 Test set: {len(X_test)} rows")
print(f"📊 Inference set: {len(X_inference)} rows")

print(f"\n📈 Class distribution in TRAINING set:")
print(y_train.value_counts())
train_pos = y_train.sum()
print(f"   → Positive cases: {train_pos} ({train_pos/len(y_train)*100:.2f}%)")

print(f"\n📈 Class distribution in TEST set:")
print(y_test.value_counts())
test_pos = y_test.sum()
print(f"   → Positive cases: {test_pos} ({test_pos/len(y_test)*100:.2f}%)")

# CRITICAL CHECKS
if test_pos == 0:
    raise ValueError(
        "❌ CRITICAL ERROR: Test set has NO positive cases!\n"
        "This will cause ROC-AUC to be NaN and metrics to be meaningless.\n"
        "Go back to preprocessing notebook and fix the train/test split."
    )

if train_pos == 0:
    raise ValueError(
        "❌ CRITICAL ERROR: Training set has NO positive cases!\n"
        "Cannot train a model without examples of crime occurring."
    )

print("\n✅ VALIDATION PASSED: Both sets have positive cases")

# ============================================================================
# 4. TRAIN LIGHTGBM MODEL
# ============================================================================
print("\n" + "="*70)
print(f"TRAINING LIGHTGBM MODEL FOR {selected_crime.upper()} - FIXED VERSION")
print("="*70)

# Calculate proper scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

print(f"\n📊 Class Distribution:")
print(f"   Negative cases: {neg_count}")
print(f"   Positive cases: {pos_count}")
print(f"   Calculated scale_pos_weight: {scale_weight:.2f}")

# Initialize LightGBM with proper parameters for imbalanced data
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=8,
    min_child_samples=20,
    scale_pos_weight=scale_weight,  # Use calculated weight
    random_state=42,
    boosting_type='gbdt',
    objective='binary',
    metric='auc',
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1
)

# Train with early stopping
print("\n🔄 Training model...")
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

print("\n✅ Training complete!")

# ============================================================================
# 5. FIND OPTIMAL THRESHOLD
# ============================================================================

# Get probability predictions
lgbm_probs = lgbm_model.predict_proba(X_test)[:, 1]

# Test multiple thresholds to find the best one
print("\n" + "="*70)
print("FINDING OPTIMAL THRESHOLD")
print("="*70)

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
best_f1 = 0
best_threshold = 0.5

print("\nThreshold | Precision | Recall | F1-Score")
print("-" * 50)

for thresh in thresholds:
    preds = (lgbm_probs >= thresh).astype(int)

    if preds.sum() > 0:  # Only if model predicts some positives
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)

        print(f"{thresh:5.2f}     | {prec:7.4f}   | {rec:6.4f} | {f1:8.4f}")

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
    else:
        print(f"{thresh:5.2f}     | No predictions made")

print(f"\n🎯 Best threshold: {best_threshold} (F1={best_f1:.4f})")

# Use the best threshold for final predictions
lgbm_preds = (lgbm_probs >= best_threshold).astype(int)







"""
print("\n" + "="*70)
print("PREPARING DATA FOR MODEL TRAINING")
print("="*70)

# 4. SELECT THE SPECIFIC CRIME (Dashboard dropdown simulation)
selected_crime = 'theft'  # Change this to train different crime models
target_col = f'target_{selected_crime}_next_week'

print(f"\n🎯 Selected crime type: {selected_crime.upper()}")
print(f"🎯 Target column: {target_col}")

# Verify target column exists
if target_col not in train_df.columns:
    raise ValueError(
        f"❌ Target column '{target_col}' not found!\n"
        f"Available target columns: {[col for col in train_df.columns if 'target_' in col]}"
    )

# 5. Extract X (features) and y (target)
X_train = train_df[features]
y_train = train_df[target_col]

X_test = test_df[features]
y_test = test_df[target_col]

X_inference = inference_full[features]

# ============================================================================
# CRITICAL VALIDATION BEFORE TRAINING
# ============================================================================

print("\n" + "="*70)
print("DATA VALIDATION BEFORE TRAINING")
print("="*70)

print(f"\n📊 Training set: {len(X_train)} rows")
print(f"📊 Test set: {len(X_test)} rows")
print(f"📊 Inference set: {len(X_inference)} rows")

print(f"\n📈 Class distribution in TRAINING set:")
print(y_train.value_counts())
train_pos = y_train.sum()
print(f"   → Positive cases: {train_pos} ({train_pos/len(y_train)*100:.2f}%)")

print(f"\n📈 Class distribution in TEST set:")
print(y_test.value_counts())
test_pos = y_test.sum()
print(f"   → Positive cases: {test_pos} ({test_pos/len(y_test)*100:.2f}%)")

# CRITICAL CHECKS
if test_pos == 0:
    raise ValueError(
        "❌ CRITICAL ERROR: Test set has NO positive cases!\n"
        "This will cause ROC-AUC to be NaN and metrics to be meaningless.\n"
        "Go back to preprocessing notebook and fix the train/test split."
    )

if train_pos == 0:
    raise ValueError(
        "❌ CRITICAL ERROR: Training set has NO positive cases!\n"
        "Cannot train a model without examples of crime occurring."
    )

print("\n✅ VALIDATION PASSED: Both sets have positive cases")
print("="*70)

# ============================================================================
# NOW PROCEED WITH MODEL TRAINING
# ============================================================================

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

print("\n" + "="*70)
print(f"TRAINING LIGHTGBM MODEL FOR {selected_crime.upper()}")
print("="*70)

# Initialize LightGBM with optimized parameters
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=24,
    scale_pos_weight=2,  # Since we used 1:3 balanced sampling
    random_state=42,
    boosting_type='gbdt',
    objective='binary',
    metric='auc'
)

# Train with early stopping
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=50)
    ]
)

# Generate predictions
lgbm_preds = (lgbm_model.predict_proba(X_test)[:, 1] > 0.5).astype(int)
lgbm_probs = lgbm_model.predict_proba(X_test)[:, 1]
inference_probs = lgbm_model.predict_proba(X_inference)[:, 1]

"""


PREPARING DATA FOR MODEL TRAINING

🎯 Selected crime type: THEFT
🎯 Target column: target_theft_next_week

DATA VALIDATION BEFORE TRAINING

📊 Training set: 2764 rows
📊 Test set: 4373 rows
📊 Inference set: 105 rows

📈 Class distribution in TRAINING set:
target_theft_next_week
0.0    2073
1.0     691
Name: count, dtype: int64
   → Positive cases: 691.0 (25.00%)

📈 Class distribution in TEST set:
target_theft_next_week
0.0    4251
1.0     122
Name: count, dtype: int64
   → Positive cases: 122.0 (2.79%)

✅ VALIDATION PASSED: Both sets have positive cases

TRAINING LIGHTGBM MODEL FOR THEFT - FIXED VERSION

📊 Class Distribution:
   Negative cases: 2073
   Positive cases: 691
   Calculated scale_pos_weight: 3.00

🔄 Training model...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 691, number of negative: 2073
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001722 seconds.
You can set `

'\nprint("\n" + "="*70)\nprint("PREPARING DATA FOR MODEL TRAINING")\nprint("="*70)\n\n# 4. SELECT THE SPECIFIC CRIME (Dashboard dropdown simulation)\nselected_crime = \'theft\'  # Change this to train different crime models\ntarget_col = f\'target_{selected_crime}_next_week\'\n\nprint(f"\n🎯 Selected crime type: {selected_crime.upper()}")\nprint(f"🎯 Target column: {target_col}")\n\n# Verify target column exists\nif target_col not in train_df.columns:\n    raise ValueError(\n        f"❌ Target column \'{target_col}\' not found!\n"\n        f"Available target columns: {[col for col in train_df.columns if \'target_\' in col]}"\n    )\n\n# 5. Extract X (features) and y (target)\nX_train = train_df[features]\ny_train = train_df[target_col]\n\nX_test = test_df[features]\ny_test = test_df[target_col]\n\nX_inference = inference_full[features]\n\n# ============================================================================\n# CRITICAL VALIDATION BEFORE TRAINING\n# ==============================

In [12]:
# ============================================================================
# 6. COMPREHENSIVE EVALUATION
# ============================================================================
print("\n" + "="*70)
print(f"MODEL EVALUATION: {selected_crime.upper()}")
print("="*70)

print("\n📊 Classification Report:")
print(classification_report(y_test, lgbm_preds, zero_division=0))

print("\n📊 Key Metrics:")
auc_score = roc_auc_score(y_test, lgbm_probs)
prec = precision_score(y_test, lgbm_preds, zero_division=0)
rec = recall_score(y_test, lgbm_preds, zero_division=0)
f1 = f1_score(y_test, lgbm_preds, zero_division=0)

print(f"   ROC-AUC Score: {auc_score:.4f}")
print(f"   Precision:     {prec:.4f}")
print(f"   Recall:        {rec:.4f}")
print(f"   F1 Score:      {f1:.4f}")

# Confusion Matrix
print("\n📊 Confusion Matrix:")
cm = confusion_matrix(y_test, lgbm_preds)
print(cm)
print(f"\n   True Negatives:  {cm[0,0]}")
print(f"   False Positives: {cm[0,1]}")
print(f"   False Negatives: {cm[1,0]}")
print(f"   True Positives:  {cm[1,1]}")

# Check predictions distribution
print(f"\n📊 Predictions Distribution:")
print(f"   Predicted No Theft: {(lgbm_preds == 0).sum()} ({(lgbm_preds == 0).sum()/len(lgbm_preds)*100:.1f}%)")
print(f"   Predicted Theft:    {(lgbm_preds == 1).sum()} ({(lgbm_preds == 1).sum()/len(lgbm_preds)*100:.1f}%)")




"""
print("\n" + "="*70)
print(f"MODEL EVALUATION: {selected_crime.upper()}")
print("="*70)

print("\n📊 Classification Report:")
print(classification_report(y_test, lgbm_preds))

print("\n📊 Key Metrics:")
auc_score = roc_auc_score(y_test, lgbm_probs)
print(f"   ROC-AUC Score: {auc_score:.4f}")
print(f"   Precision:     {precision_score(y_test, lgbm_preds):.4f}")
print(f"   Recall:        {recall_score(y_test, lgbm_preds):.4f}")
print(f"   F1 Score:      {f1_score(y_test, lgbm_preds):.4f}")

# Check if results are valid
if pd.isna(auc_score):
    print("\n❌ WARNING: ROC-AUC is NaN - this indicates test set has only one class!")
else:
    print("\n✅ Model evaluation successful!")

print("\n" + "="*70)
print("INFERENCE PREDICTIONS FOR HEATMAP")
print("="*70)
print(f"Generated {len(inference_probs)} risk scores for GN divisions")
print(f"Risk score range: {inference_probs.min():.3f} to {inference_probs.max():.3f}")
"""



MODEL EVALUATION: THEFT

📊 Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      4251
         1.0       0.63      0.66      0.65       122

    accuracy                           0.98      4373
   macro avg       0.81      0.83      0.82      4373
weighted avg       0.98      0.98      0.98      4373


📊 Key Metrics:
   ROC-AUC Score: 0.9345
   Precision:     0.6279
   Recall:        0.6639
   F1 Score:      0.6454

📊 Confusion Matrix:
[[4203   48]
 [  41   81]]

   True Negatives:  4203
   False Positives: 48
   False Negatives: 41
   True Positives:  81

📊 Predictions Distribution:
   Predicted No Theft: 4244 (97.1%)
   Predicted Theft:    129 (2.9%)


'\nprint("\n" + "="*70)\nprint(f"MODEL EVALUATION: {selected_crime.upper()}")\nprint("="*70)\n\nprint("\n📊 Classification Report:")\nprint(classification_report(y_test, lgbm_preds))\n\nprint("\n📊 Key Metrics:")\nauc_score = roc_auc_score(y_test, lgbm_probs)\nprint(f"   ROC-AUC Score: {auc_score:.4f}")\nprint(f"   Precision:     {precision_score(y_test, lgbm_preds):.4f}")\nprint(f"   Recall:        {recall_score(y_test, lgbm_preds):.4f}")\nprint(f"   F1 Score:      {f1_score(y_test, lgbm_preds):.4f}")\n\n# Check if results are valid\nif pd.isna(auc_score):\n    print("\n❌ WARNING: ROC-AUC is NaN - this indicates test set has only one class!")\nelse:\n    print("\n✅ Model evaluation successful!")\n\nprint("\n" + "="*70)\nprint("INFERENCE PREDICTIONS FOR HEATMAP")\nprint("="*70)\nprint(f"Generated {len(inference_probs)} risk scores for GN divisions")\nprint(f"Risk score range: {inference_probs.min():.3f} to {inference_probs.max():.3f}")\n'